# FK & Throw-in export — every competition with season_id 318

Produces two parquet files per league (written to `Data/SP/`):
- `{league} - Freekicks.parquet`  — attacking-half free-kick restarts (x ≥ 60)
- `{league} - Throwins.parquet`  — attacking-third throw-ins that enter the box within 3 actions **or** middle-third throw-ins that retain possession within 3 actions

In [1]:
from __future__ import annotations

import os
import subprocess
import tempfile
from pathlib import Path
from typing import Callable

import pandas as pd
from statsbombpy import sb


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DATA_DIR = Path("/Users/user/Documents/GitHub/mm-setpieces-1/Data")
OUTPUT_DIR    = BASE_DATA_DIR / "SP"
SWE_SP_TEMPLATE = BASE_DATA_DIR / "SP" / "SWE SP.xlsx"

# ── Season ─────────────────────────────────────────────────────────────────────
TARGET_SEASON_ID = 318

# ── Credentials ────────────────────────────────────────────────────────────────
ACCOUNTS = [
    {"user": "m.pulley@az.nl",          "passwd": "SwHrVcks"},
    {"user": "JACK71299@HOTMAIL.CO.UK", "passwd": "J7rB7aP2"},
]
ACCOUNTS = [a for a in ACCOUNTS if a.get("user") and a.get("passwd")]

# ── Geometry constants ─────────────────────────────────────────────────────────
PENALTY_BOX_X      = 102
PENALTY_BOX_Y_MIN  = 18
PENALTY_BOX_Y_MAX  = 62
ATTACKING_HALF_X   = 60
ATTACKING_THIRD_X  = 80
MIDDLE_THIRD_MIN_X = 40
NEXT_ACTIONS       = 3

WANTED_PATTERNS = {"From Throw In", "From Free Kick"}

In [4]:
# ── Logging & I/O ──────────────────────────────────────────────────────────────

def log(message: str) -> None:
    print(message, flush=True)


def save_parquet(df: pd.DataFrame, file_name: str) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_file = OUTPUT_DIR / file_name
    try:
        df.to_parquet(output_file, index=False)
        return
    except Exception as exc:
        print(f"Default parquet save failed ({exc}). Trying bundled parquet writer...")

    bundled_python   = "/Users/user/.cache/codex-runtimes/codex-primary-runtime/dependencies/python/bin/python3"
    bundled_packages = "/Users/user/Documents/Codex/2026-05-24/files-mentioned-by-the-user-corners-3/.python-packages"
    if not os.path.exists(bundled_python) or not os.path.exists(bundled_packages):
        raise

    writer_code = r"""
import sys, pandas as pd
pickle_file, output_file = sys.argv[1], sys.argv[2]
df = pd.read_pickle(pickle_file)
df.to_parquet(output_file, index=False, engine='pyarrow')
"""
    with tempfile.TemporaryDirectory() as tmpdir:
        pickle_file = os.path.join(tmpdir, "fk_throws.pkl")
        df.to_pickle(pickle_file)
        env = os.environ.copy()
        env["PYTHONPATH"] = bundled_packages + os.pathsep + env.get("PYTHONPATH", "")
        subprocess.run([bundled_python, "-c", writer_code, pickle_file, str(output_file)],
                       check=True, env=env)

In [5]:
# ── Account rotation ───────────────────────────────────────────────────────────

def try_accounts(fetcher: Callable, *args, **kwargs):
    for account in ACCOUNTS:
        try:
            out = fetcher(account, *args, **kwargs)
        except Exception as exc:
            log(f"  Account {account['user']} failed: {exc}")
            continue
        if out is None:
            continue
        if isinstance(out, pd.DataFrame) and out.empty:
            continue
        if not isinstance(out, pd.DataFrame) and len(out) == 0:
            continue
        return out
    return None


def get_competitions_one(account: dict) -> pd.DataFrame | None:
    comps = sb.competitions(creds=account)
    return None if comps is None or comps.empty else comps


def get_matches_one(account: dict, season_id: int, comp_id: int) -> pd.DataFrame | None:
    matches = sb.matches(competition_id=comp_id, season_id=season_id, creds=account)
    return None if matches is None or matches.empty else matches


def get_events_one(account: dict, match_id: int) -> pd.DataFrame | None:
    events = sb.events(match_id=match_id, creds=account)
    if events is None or events.empty:
        return None
    events = events.copy()
    events["match_id"] = match_id
    return events

In [6]:
# ── Value helpers ──────────────────────────────────────────────────────────────

def pick_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def value_from(row: pd.Series, candidates: list[str | None], default=None):
    for c in candidates:
        if c is None or c not in row.index:
            continue
        v = row[c]
        if isinstance(v, (list, tuple, dict)):
            return v
        if pd.notna(v):
            return v
    return default


def text_value(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    if isinstance(value, dict):
        for key in ["name", "type", "id"]:
            if key in value:
                return str(value[key])
        return ", ".join(str(v) for v in value.values() if pd.notna(v))
    return str(value)


def stringify(value) -> str | None:
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    if isinstance(value, (list, tuple)):
        return ", ".join(stringify(item) or "" for item in value)
    if isinstance(value, dict):
        parts = [stringify(v) for v in value.values()]
        return ", ".join(p for p in parts if p)
    return str(value)


def location_xy(value) -> tuple[float | None, float | None]:
    if isinstance(value, (list, tuple)) and len(value) >= 2:
        try:
            return float(value[0]), float(value[1])
        except (TypeError, ValueError):
            return None, None
    if isinstance(value, str):
        parts = [p.strip() for p in value.split(",")]
        if len(parts) >= 2:
            try:
                return float(parts[0]), float(parts[1])
            except ValueError:
                return None, None
    return None, None


def in_penalty_box(value) -> bool:
    x, y = location_xy(value)
    return (
        x is not None and y is not None
        and x >= PENALTY_BOX_X
        and PENALTY_BOX_Y_MIN <= y <= PENALTY_BOX_Y_MAX
    )

In [7]:
# ── Set-piece column schema ────────────────────────────────────────────────────

def swe_sp_columns() -> list[str]:
    columns = []
    if SWE_SP_TEMPLATE.exists():
        try:
            columns = list(pd.read_excel(SWE_SP_TEMPLATE, nrows=0).columns)
        except Exception as exc:
            log(f"Could not read SWE SP template columns: {exc}")

    if not columns:
        columns = [
            "match_id", "possession", "team.name", "type.name", "SP_Type", "location.pass",
            "pass.height.name", "timestamp", "Taker", "Shooter", "location.shot",
            "shot.statsbomb_xg", "shot.freeze_frame", "shot.outcome.name", "shot_x", "shot_y",
            "Metrics", "Occupation_Rating", "Proximity_Rating", "Duel_Win_Prob", "OPS_Opponent_Rating",
        ]

    extra_columns = [
        "Restart_Profile", "Start_Third", "Next_3_Box_Entry", "Next_3_Retain_Possession",
        "restart_x", "restart_y", "actions_checked", "delivery_end_x", "delivery_end_y",
    ]
    return columns + [c for c in extra_columns if c not in columns]

In [8]:
# ── Next-action analysis ───────────────────────────────────────────────────────

def sorted_events(events: pd.DataFrame) -> pd.DataFrame:
    sort_cols = [c for c in ["period", "timestamp", "index"] if c in events.columns]
    return events.sort_values(sort_cols).reset_index(drop=True) if sort_cols else events.reset_index(drop=True)


def next_actions(events: pd.DataFrame, pos: int) -> pd.DataFrame:
    return events.iloc[pos + 1 : pos + 1 + NEXT_ACTIONS].copy()


def next_actions_enter_box(actions: pd.DataFrame) -> bool:
    location_candidates = [
        "location", "pass_end_location", "pass.end_location",
        "carry_end_location", "carry.end_location",
        "shot_end_location", "shot.end_location", "location.shot",
    ]
    for _, action in actions.iterrows():
        for col in location_candidates:
            if col in action.index and in_penalty_box(action[col]):
                return True
    return False


def first_box_entry_location(actions: pd.DataFrame) -> tuple[float | None, float | None]:
    location_candidates = [
        "location", "pass_end_location", "pass.end_location",
        "carry_end_location", "carry.end_location",
        "shot_end_location", "shot.end_location", "location.shot",
    ]
    for _, action in actions.iterrows():
        for col in location_candidates:
            if col in action.index and in_penalty_box(action[col]):
                return location_xy(action[col])
    return None, None


def next_actions_retain_possession(actions: pd.DataFrame, possession, team_name: str) -> bool:
    if actions.empty:
        return False
    if "possession" in actions.columns and possession is not None:
        return actions["possession"].eq(possession).all()
    team_col = pick_column(actions, ["team", "team.name"])
    if team_col and team_name:
        return actions[team_col].map(text_value).eq(team_name).all()
    return False


def restart_profile(sp_type: str, start_x: float | None, entered_box: bool, retained: bool) -> str | None:
    if sp_type == "From Free Kick":
        if start_x is not None and start_x >= ATTACKING_HALF_X:
            return "Attacking half free kick"
        return None
    if sp_type == "From Throw In":
        if start_x is not None and start_x >= ATTACKING_THIRD_X and entered_box:
            return "Attacking third attack"
        if start_x is not None and MIDDLE_THIRD_MIN_X <= start_x < ATTACKING_THIRD_X and retained:
            return "Middle third retained"
        return None
    return None


def sequence_delivery_end(
    actions: pd.DataFrame,
    entered_box: bool,
    shot_x: float | None,
    shot_y: float | None,
    pass_end_location,
) -> tuple[float | None, float | None]:
    if shot_x is not None:
        return shot_x, shot_y
    if entered_box:
        x, y = first_box_entry_location(actions)
        if x is not None:
            return x, y
    return location_xy(pass_end_location)

In [9]:
# ── Core formatter ─────────────────────────────────────────────────────────────

def format_swe_sp_like(events: pd.DataFrame) -> pd.DataFrame:
    events = sorted_events(events)
    play_pattern_col = pick_column(events, ["play_pattern", "play_pattern.name"])
    type_col         = pick_column(events, ["type", "type_name", "type.name"])
    possession_col   = pick_column(events, ["possession"])
    pass_type_col    = pick_column(events, ["pass_type", "pass.type.name"])
    pass_height_col  = pick_column(events, ["pass_height", "pass.height.name"])
    pass_end_col     = pick_column(events, ["pass_end_location", "pass.end_location"])
    shot_outcome_col = pick_column(events, ["shot_outcome", "shot.outcome.name"])
    xg_col           = pick_column(events, ["shot_statsbomb_xg", "shot.statsbomb_xg"])

    if play_pattern_col is None or type_col is None:
        return pd.DataFrame(columns=swe_sp_columns())

    rows = []
    event_type_text  = events[type_col].map(text_value)
    play_pattern_text = events[play_pattern_col].map(text_value)
    pass_type_text   = events[pass_type_col].map(text_value) if pass_type_col else pd.Series("", index=events.index)

    restart_mask = (
        event_type_text.str.lower().eq("pass")
        & play_pattern_text.isin(WANTED_PATTERNS)
    )
    if pass_type_col:
        restart_mask = restart_mask & pass_type_text.isin(["Free Kick", "Throw-in", "Throw In"])

    for event_position, event in events.loc[restart_mask].iterrows():
        sp_type       = text_value(event[play_pattern_col])
        possession    = value_from(event, ["possession"])
        team_name     = text_value(value_from(event, ["team", "team.name"]))
        pass_location = value_from(event, ["location", "location.pass"])
        start_x, start_y = location_xy(pass_location)
        pass_end_location = value_from(event, [pass_end_col]) if pass_end_col else None
        actions       = next_actions(events, event_position)
        entered_box   = next_actions_enter_box(actions)
        retained      = next_actions_retain_possession(actions, possession, team_name)
        profile       = restart_profile(sp_type, start_x, entered_box, retained)

        if profile is None:
            continue

        if possession_col:
            possession_events = events[events[possession_col] == possession].copy()
            if "index" in possession_events.columns and "index" in event.index:
                possession_events = possession_events[possession_events["index"] >= event["index"]]
        else:
            possession_events = events.iloc[0:0].copy()

        shot_events = possession_events[
            possession_events[type_col].map(text_value).str.lower().eq("shot")
        ].copy()
        shot = shot_events.iloc[0] if not shot_events.empty else pd.Series(dtype="object")
        shot_location = value_from(shot, ["location", "location.shot", "shot_end_location"])
        shot_x, shot_y = location_xy(shot_location)

        delivery_end_x, delivery_end_y = sequence_delivery_end(
            actions, entered_box, shot_x, shot_y, pass_end_location
        )

        if start_x is None:
            start_third = "Unknown"
        elif start_x >= ATTACKING_THIRD_X:
            start_third = "Attacking third"
        elif start_x >= MIDDLE_THIRD_MIN_X:
            start_third = "Middle third"
        else:
            start_third = "Defensive third"

        rows.append({
            "match_id":                  value_from(event, ["match_id"]),
            "possession":                possession,
            "team.name":                 team_name,
            "type.name":                 text_value(value_from(event, [type_col])),
            "SP_Type":                   sp_type,
            "location.pass":             stringify(pass_location),
            "pass.height.name":          value_from(event, [pass_height_col]) if pass_height_col else None,
            "timestamp":                 value_from(event, ["timestamp"]),
            "Taker":                     value_from(event, ["player", "player.name"]),
            "Shooter":                   value_from(shot, ["player", "player.name"]),
            "location.shot":             stringify(shot_location),
            "shot.statsbomb_xg":         value_from(shot, [xg_col]) if xg_col else None,
            "shot.freeze_frame":         stringify(value_from(shot, ["shot_freeze_frame", "shot.freeze_frame"])),
            "shot.outcome.name":         value_from(shot, [shot_outcome_col]) if shot_outcome_col else None,
            "shot_x":                    shot_x,
            "shot_y":                    shot_y,
            "Metrics":                   "0, Inf, 1",
            "Occupation_Rating":         0,
            "Proximity_Rating":          pd.NA,
            "Duel_Win_Prob":             1,
            "OPS_Opponent_Rating":       0,
            "Restart_Profile":           profile,
            "Start_Third":               start_third,
            "Next_3_Box_Entry":          entered_box,
            "Next_3_Retain_Possession":  retained,
            "restart_x":                 start_x,
            "restart_y":                 start_y,
            "actions_checked":           len(actions),
            "delivery_end_x":            delivery_end_x,
            "delivery_end_y":            delivery_end_y,
        })

    output = pd.DataFrame(rows)
    columns = swe_sp_columns()
    for col in columns:
        if col not in output.columns:
            output[col] = pd.NA
    return output[columns]

In [10]:
# ── Per-league runner ──────────────────────────────────────────────────────────

def run_league(league: dict) -> None:
    name    = league["name"]
    comp_id = league["competition_id"]
    season_id = league["season_id"]

    log(f"\n{'='*60}")
    log(f"League: {name}  (competition={comp_id}, season={season_id})")
    log(f"{'='*60}")

    log("Fetching matches...")
    matches_df = try_accounts(get_matches_one, season_id=season_id, comp_id=comp_id)
    if matches_df is None or matches_df.empty or "match_id" not in matches_df.columns:
        log(f"No matches found for {name} — skipping.")
        return

    match_ids = matches_df["match_id"].dropna().drop_duplicates().astype(int).tolist()
    log(f"Found {len(match_ids)} matches.")

    filtered_events_list = []
    for match_id in match_ids:
        log(f"  Processing match {match_id}...")
        events = try_accounts(get_events_one, match_id=match_id)
        if events is not None and not events.empty:
            filtered = format_swe_sp_like(events)
            if not filtered.empty:
                filtered_events_list.append(filtered)
        else:
            log(f"  No events for match {match_id}")

    if filtered_events_list:
        all_events = pd.concat(filtered_events_list, ignore_index=True, sort=False)
    else:
        all_events = pd.DataFrame(columns=swe_sp_columns())

    if "SP_Type" in all_events.columns:
        freekicks = all_events[all_events["SP_Type"].eq("From Free Kick")].copy()
        throwins  = all_events[all_events["SP_Type"].eq("From Throw In")].copy()
    else:
        freekicks = pd.DataFrame(columns=swe_sp_columns())
        throwins  = pd.DataFrame(columns=swe_sp_columns())

    save_parquet(freekicks, league["freekicks_file"])
    save_parquet(throwins,  league["throwins_file"])
    log(f"Saved {league['freekicks_file']} ({len(freekicks)} rows)")
    log(f"Saved {league['throwins_file']} ({len(throwins)} rows)")

In [11]:
# ── Discover every competition for season_id 318 ──────────────────────────────
#
# Competitions that share a name get the competition_id appended to stay unique.

if not ACCOUNTS:
    raise SystemExit("No credentials found.")

log("Fetching competition list...")
all_comps = try_accounts(get_competitions_one)
if all_comps is None or all_comps.empty:
    raise SystemExit("Could not retrieve competitions from StatsBomb.")

season_comps = (
    all_comps[all_comps["season_id"] == TARGET_SEASON_ID]
    [["competition_id", "competition_name"]]
    .drop_duplicates()
    .sort_values("competition_name")
    .reset_index(drop=True)
)

log(f"Found {len(season_comps)} competitions for season_id={TARGET_SEASON_ID}:")
print(season_comps.to_string(index=False))

# Build unique display names (append competition_id for duplicates)
name_counts = season_comps["competition_name"].value_counts()

def unique_name(row) -> str:
    base = row["competition_name"]
    return f"{base} ({row['competition_id']})" if name_counts[base] > 1 else base

season_comps["unique_name"] = season_comps.apply(unique_name, axis=1)

# Sanitise for file names (strip characters forbidden on Windows / macOS)
def safe_filename(name: str) -> str:
    for ch in r'\/:*?"<>|':
        name = name.replace(ch, "_")
    return name.strip()

LEAGUES = [
    {
        "name":           row["unique_name"],
        "competition_id": int(row["competition_id"]),
        "season_id":      TARGET_SEASON_ID,
        "freekicks_file": f"{safe_filename(row['unique_name'])} - Freekicks.parquet",
        "throwins_file":  f"{safe_filename(row['unique_name'])} - Throwins.parquet",
    }
    for _, row in season_comps.iterrows()
]

log(f"\nWill write to: {OUTPUT_DIR}")

Fetching competition list...
Found 78 competitions for season_id=318:
 competition_id                         competition_name
              9                            1. Bundesliga
            260                              1. Division
             78                                   1. HNL
           1714                                   1. SNL
             10                            2. Bundesliga
           1581                                  2. Liga
            179                                  3. Liga
             93                                 A-League
           1611                    AZ Academy Friendlies
           1527                            AZ Friendlies
           1551                                  AZ U15s
           1552                                  AZ U16s
           1555                                  AZ U17s
           1746                                  AZ U19s
             47                               Bundesliga
            121   

In [12]:
# ── Run all leagues ────────────────────────────────────────────────────────────

for league in LEAGUES:
    run_league(league)

log("\nAll leagues done.")


League: 1. Bundesliga  (competition=9, season=318)
Fetching matches...
Found 306 matches.
  Processing match 4003398...
  Processing match 4003470...


/opt/anaconda3/lib/python3.12/site-packages/statsbombpy/sb.py:164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([*events.values()], axis=0, ignore_index=True, sort=True)


  Processing match 4003323...


/opt/anaconda3/lib/python3.12/site-packages/statsbombpy/sb.py:164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([*events.values()], axis=0, ignore_index=True, sort=True)
/opt/anaconda3/lib/python3.12/site-packages/statsbombpy/sb.py:164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([*events.values()], axis=0, ignore_index=True, sort=True)


  Processing match 4003467...
  Processing match 4003311...


...

  Processing match 4029281...


/opt/anaconda3/lib/python3.12/site-packages/statsbombpy/sb.py:164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([*events.values()], axis=0, ignore_index=True, sort=True)
/opt/anaconda3/lib/python3.12/site-packages/statsbombpy/sb.py:164: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([*events.values()], axis=0, ignore_index=True, sort=True)


  Processing match 4011319...
  Account m.pulley@az.nl failed: disk I/O error
  Account JACK71299@HOTMAIL.CO.UK failed: disk I/O error
  No events for match 4011319
  Processing match 4011358...
  Account m.pulley@az.nl failed: disk I/O error
  Account JACK71299@HOTMAIL.CO.UK failed: disk I/O error
  No events for match 4011358
  Processing match 4029511...
  Account m.pulley@az.nl failed: disk I/O error
  Account JACK71299@HOTMAIL.CO.UK failed: disk I/O error
  No events for match 4029511
  Processing match 4011348...
  Account m.pulley@az.nl failed: disk I/O error
  Account JACK71299@HOTMAIL.CO.UK failed: disk I/O error
  No events for match 4011348
  Processing match 4029380...
  Account m.pulley@az.nl failed: disk I/O error
  Account JACK71299@HOTMAIL.CO.UK failed: disk I/O error
  No events for match 4029380
  Processing match 4011339...
  Account m.pulley@az.nl failed: disk I/O error
  Account JACK71299@HOTMAIL.CO.UK failed: disk I/O error
  No events for match 4011339
  Processi

/var/folders/vh/gd9x5_d94g33w411t0md800c0000gn/T/ipykernel_1679/2356292071.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_events = pd.concat(filtered_events_list, ignore_index=True, sort=False)


OSError: [Errno 28] No space left on device: '/var/folders/vh/gd9x5_d94g33w411t0md800c0000gn/T/tmp1ibvakkt'